In [1]:
# ==========================================
# Notebook 2: Create the Labels
# ==========================================

import os
import pandas as pd

# 1. Chargement de l'artefact généré au Notebook 1
input_path = "artifacts/01_joined_ml_table.parquet"
df = pd.read_parquet(input_path)
print(f"Dataset chargé avec succès : {df.shape[0]} lignes.")

# 2. Conversion des colonnes de dates au format datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

Dataset chargé avec succès : 96478 lignes.


In [2]:
# 3. Création du label binaire de retard (1 = En retard, 0 = Dans les temps)
# Une commande est en retard si la livraison effective dépasse la date estimée (au niveau de la date brute)
df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)

# Calcul du délai en jours (positif = jours de retard, négatif = jours d'avance)
df['delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / 86400

# 4. Vérification sur quelques commandes réelles
sample_cols = [
    'order_id', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date', 
    'delay_days', 
    'is_late'
]

print("\n=== ÉCHANTILLON DE COMMANDES À L'HEURE (is_late = 0) ===")
print(df[df['is_late'] == 0][sample_cols].head(3).to_string())

print("\n=== ÉCHANTILLON DE COMMANDES EN RETARD (is_late = 1) ===")
print(df[df['is_late'] == 1][sample_cols].head(3).to_string())


=== ÉCHANTILLON DE COMMANDES À L'HEURE (is_late = 0) ===
                           order_id order_delivered_customer_date order_estimated_delivery_date  delay_days  is_late
0  e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:13                    2017-10-18   -7.107488        0
1  53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:45                    2018-08-13   -5.355729        0
2  47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:29                    2018-09-04  -17.245498        0

=== ÉCHANTILLON DE COMMANDES EN RETARD (is_late = 1) ===
                            order_id order_delivered_customer_date order_estimated_delivery_date  delay_days  is_late
19  203096f03d82e0dffbc41ebc2e2bcfb7           2017-10-09 22:23:46                    2017-09-28   11.933171        1
24  fbf9ac61453ac646ce8ad9783d7d0af6           2018-03-21 22:03:54                    2018-03-12    9.919375        1
34  8563039e855156e48fccee4d611a3196           2018-03-20 00:5

In [3]:
# 1. Analyse de la distribution des classes
class_counts = df['is_late'].value_counts()
class_percentages = df['is_late'].value_counts(normalize=True) * 100

print("\n=== DISTRIBUTION DES CLASSES ===")
print(f"A l'heure (0) : {class_counts.get(0, 0):>6} commandes ({class_percentages.get(0, 0):.2f}%)")
print(f"En retard (1) : {class_counts.get(1, 0):>6} commandes ({class_percentages.get(1, 0):.2f}%)")

# 2. Évaluation du déséquilibre de classe (Class Imbalance)
late_ratio = class_percentages.get(1, 0)
print("\n=== DIAGNOSTIC DU DESÉQUILIBRE DE CLASSE ===")
if late_ratio < 10:
    print(f"Déséquilibre SÉVÈRE ({late_ratio:.2f}% de retards). Stratégies recommandées : PR-AUC, F1-Score, SMOTE, class_weight='balanced'.")
elif late_ratio < 20:
    print(f"Déséquilibre MODÉRÉ ({late_ratio:.2f}% de retards). Ajuster 'scale_pos_weight' / 'class_weight' et privilégier PR-AUC/ROC-AUC.")
else:
    print(f"Déséquilibre LÉGER ({late_ratio:.2f}% de retards). Un entraînement standard reste utilisable.")


=== DISTRIBUTION DES CLASSES ===
A l'heure (0) :  88652 commandes (91.89%)
En retard (1) :   7826 commandes (8.11%)

=== DIAGNOSTIC DU DESÉQUILIBRE DE CLASSE ===
Déséquilibre SÉVÈRE (8.11% de retards). Stratégies recommandées : PR-AUC, F1-Score, SMOTE, class_weight='balanced'.


In [4]:
# 1. Sauvegarde de l'artefact étiqueté
output_path = "artifacts/02_labeled_ml_table.parquet"
df.to_parquet(output_path, index=False)
print(f"\n✅ Artefact N°2 sauvegardé dans : '{output_path}'")


✅ Artefact N°2 sauvegardé dans : 'artifacts/02_labeled_ml_table.parquet'
